# PySpark CDC — SCD Type 1 Current State

Current-state CDC example. Latest CDC event per key wins. Inserts and updates become the new current row. Deletes remove the current row.

## 00 — Spark setup and sample data

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("cdc-scd1-merge")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/cdc-scd1-merge_warehouse")
    .getOrCreate()
)

spark.version

'4.0.2'

In [2]:
customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("updated_at", StringType(), False),
])

cdc_events_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("op", StringType(), False),
    StructField("event_time", StringType(), False),
])

customers = spark.createDataFrame([
    (1, "Alice", "alice@old.com", "2026-05-02T09:00:00"),
    (2, "Bob", "bob@mail.com", "2026-05-02T09:00:00"),
    (3, "Carol", "carol@mail.com", "2026-05-02T09:00:00"),
], customers_schema).withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)

cdc_events = spark.createDataFrame([
    (1, "Alice", "alice@new.com", "U", "2026-05-02T10:05:00"),
    (2, None, None, "D", "2026-05-02T10:10:00"),
    (4, "Diana", "diana@mail.com", "I", "2026-05-02T10:15:00"),
    (1, "Alice A.", "alice.a@mail.com", "U", "2026-05-02T10:20:00"),
], cdc_events_schema).withColumn(
    "event_time",
    F.to_timestamp("event_time")
)

print("Customers before CDC")
customers.orderBy("customer_id").show(truncate=False)

print("CDC events")
cdc_events.orderBy("event_time").show(truncate=False)

Customers before CDC


+-----------+-----+--------------+-------------------+
|customer_id|name |email         |updated_at         |
+-----------+-----+--------------+-------------------+
|1          |Alice|alice@old.com |2026-05-02 09:00:00|
|2          |Bob  |bob@mail.com  |2026-05-02 09:00:00|
|3          |Carol|carol@mail.com|2026-05-02 09:00:00|
+-----------+-----+--------------+-------------------+

CDC events
+-----------+--------+----------------+---+-------------------+
|customer_id|name    |email           |op |event_time         |
+-----------+--------+----------------+---+-------------------+
|1          |Alice   |alice@new.com   |U  |2026-05-02 10:05:00|
|2          |NULL    |NULL            |D  |2026-05-02 10:10:00|
|4          |Diana   |diana@mail.com  |I  |2026-05-02 10:15:00|
|1          |Alice A.|alice.a@mail.com|U  |2026-05-02 10:20:00|
+-----------+--------+----------------+---+-------------------+



## 01 — Latest event per key

A single CDC batch can contain several events for the same business key.  
For a current-state table, keep only the newest event per key before applying changes.

In [3]:
from pyspark.sql.window import Window

latest_event_window = Window.partitionBy("customer_id").orderBy(F.col("event_time").desc())

latest_cdc = (
    cdc_events
    .withColumn("rn", F.row_number().over(latest_event_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

latest_cdc.orderBy("customer_id").show(truncate=False)

+-----------+--------+----------------+---+-------------------+
|customer_id|name    |email           |op |event_time         |
+-----------+--------+----------------+---+-------------------+
|1          |Alice A.|alice.a@mail.com|U  |2026-05-02 10:20:00|
|2          |NULL    |NULL            |D  |2026-05-02 10:10:00|
|4          |Diana   |diana@mail.com  |I  |2026-05-02 10:15:00|
+-----------+--------+----------------+---+-------------------+



## 02 — Apply deletes

Rows with `op = 'D'` must disappear from the current-state result.

In [4]:
delete_keys = latest_cdc.filter(F.col("op") == "D").select("customer_id")

after_deletes = customers.join(delete_keys, on="customer_id", how="left_anti")

after_deletes.orderBy("customer_id").show(truncate=False)

+-----------+-----+--------------+-------------------+
|customer_id|name |email         |updated_at         |
+-----------+-----+--------------+-------------------+
|1          |Alice|alice@old.com |2026-05-02 09:00:00|
|3          |Carol|carol@mail.com|2026-05-02 09:00:00|
+-----------+-----+--------------+-------------------+



## 03 — Apply inserts and updates

For SCD Type 1, updates overwrite the previous values.

In [5]:
upserts = (
    latest_cdc
    .filter(F.col("op").isin("I", "U"))
    .select(
        "customer_id",
        "name",
        "email",
        F.col("event_time").alias("updated_at")
    )
)

unchanged_rows = after_deletes.join(
    upserts.select("customer_id"),
    on="customer_id",
    how="left_anti"
)

customers_after_cdc = unchanged_rows.unionByName(upserts)

customers_after_cdc.orderBy("customer_id").show(truncate=False)

+-----------+--------+----------------+-------------------+
|customer_id|name    |email           |updated_at         |
+-----------+--------+----------------+-------------------+
|1          |Alice A.|alice.a@mail.com|2026-05-02 10:20:00|
|3          |Carol   |carol@mail.com  |2026-05-02 09:00:00|
|4          |Diana   |diana@mail.com  |2026-05-02 10:15:00|
+-----------+--------+----------------+-------------------+



## 04 — Expected result

- Customer `1` updated to the latest event
- Customer `2` deleted
- Customer `3` unchanged
- Customer `4` inserted

In [6]:
customers_after_cdc.orderBy("customer_id").show(truncate=False)

+-----------+--------+----------------+-------------------+
|customer_id|name    |email           |updated_at         |
+-----------+--------+----------------+-------------------+
|1          |Alice A.|alice.a@mail.com|2026-05-02 10:20:00|
|3          |Carol   |carol@mail.com  |2026-05-02 09:00:00|
|4          |Diana   |diana@mail.com  |2026-05-02 10:15:00|
+-----------+--------+----------------+-------------------+

